# 05 LLM 판단 노트북

광고 1건을 넣으면 **검사기준표 30개 항목**을 판정하고, 항목마다 근거 조항과 원문 인용을 붙여 줍니다.

```
광고 JSON(주연 님 전처리)
  → ① 상품유형 정하기
  → ② 코드 판정 (숫자 기준: 대비, 글자 크기, 노출 시간, 심사필 …)
  → ③ 근거 카드 검색 (04 RAG: BM25 + BGE-M3 + 리랭커)
  → ④ LLM 판단 (문구 항목 판정 + 모든 항목에 근거 인용)
  → ⑤ 근거 검증 (인용한 카드·문장이 진짜 있는지 코드로 확인)
  → ⑥ 결과표 + 종합 의견 저장
```

**순서**
1. 준비 (API 키, 경로)
2. 검사기준표 불러오기
3. 상품유형 자동 채우기
4. 광고 불러오기 + 문구 정리
5. 코드 판정
6. 근거 카드 검색
7. 프롬프트 만들기
8. LLM 호출 (OpenAI / Claude)
9. 근거 검증
10. 광고 1건 판정
11. 여러 광고 · 여러 모델 비교

## 1. 준비
처음 한 번만 터미널에서 설치합니다 (가상환경을 켠 상태에서).
```bash
pip install openai anthropic python-dotenv
```
API 키는 `rag/.env` 파일에 넣습니다. **키를 노트북 안에 직접 쓰지 마세요.**
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...      # Claude는 나중에 추가해도 됩니다
```

In [ ]:
import json, re, datetime
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

import os
RAG_DIR = Path.cwd().resolve()            # 이 노트북이 있는 rag/ 폴더
DATA_DIR = RAG_DIR / 'data'
OUT_DIR = RAG_DIR / 'outputs'; OUT_DIR.mkdir(exist_ok=True)
load_dotenv(RAG_DIR / '.env')

def find_repo_dir():
    # 광고 JSON(data/processed/...)이 있는 MIX 레포 폴더를 찾습니다.
    # 1) .env의 MIX_REPO_DIR  2) 이 폴더와 상위 폴더 중 data/processed가 있는 곳
    env = os.getenv('MIX_REPO_DIR')
    cands = [Path(env).expanduser()] if env else []
    cands += [RAG_DIR, *RAG_DIR.parents]
    for d in cands:
        if (d / 'data/processed/image/json').is_dir() or (d / 'data/processed/video/json').is_dir():
            return d.resolve()
    raise FileNotFoundError('광고 JSON 폴더(data/processed/{image,video}/json)를 찾지 못했습니다. '
                            'rag/.env에 MIX_REPO_DIR=/path/to/MIX 를 적어 주세요.')

REPO_DIR = find_repo_dir()
print('광고 JSON 위치:', REPO_DIR / 'data/processed')
print('OpenAI 키:', '있음' if os.getenv('OPENAI_API_KEY') else '없음')
print('Claude 키:', '있음' if os.getenv('ANTHROPIC_API_KEY') else '없음')

# 개발 중에는 저가 모델, 최종 비교 때 중간 모델로 바꿉니다
MODELS = {
    'openai': 'gpt-6-luna',            # 최종 비교: 'gpt-6.2-sol'
    'claude': 'claude-haiku-5-5',      # 최종 비교: 'claude-sonnet-5-5'
}
TODAY = datetime.date.today()
pd.set_option('display.max_colwidth', 120)   # 표에서 긴 글이 너무 잘리지 않게

04 노트북에서 만든 검색 장치를 불러옵니다 (`rag_search.py`). 모델을 불러오느라 30초~1분 걸립니다.

In [ ]:
from rag_search import search_ids, card_by_id
search_ids('영상 광고에서 위험고지가 짧게만 노출됨', '투자성', k=3)

## 2. 검사기준표 불러오기
`data/checks.csv`는 Google 시트의 **검사기준표** 탭에 노트북이 쓰는 열 몇 개를 더한 것입니다.
- `products`: 적용 상품유형 (`전체`, `투자성`, `!대출성` = 대출성 제외)
- `media`: 적용 매체 (`all`, `image`, `video`, `banner` …)
- `mode`: `code`(숫자로 판정), `llm`(LLM이 판정), `code+llm`, `skip`(VLM 단계에서 판단)
- `group`: LLM에 함께 묻는 묶음 (AD: 표시 방법·심의 표시, B: 필수 내용, CE: 금지 표현·추천보증)

기준값(TH)은 검사기준표의 임시값입니다. 보정하면 여기만 고치면 됩니다.

In [ ]:
checks = pd.read_csv(DATA_DIR / 'checks.csv', encoding='utf-8-sig').fillna('')

TH = {
    'contrast': 3.0,          # A2 명암비
    'image_pt': 5.9,          # A3 인쇄형 글자 크기 (폭 390 기준, 9pt 샘플로 보정 예정)
    'ratio_image': 0.075,     # A4 이미지: 실제 광고 하위 10%
    'ratio_video': 0.10,      # A4 영상: 실제 광고 하위 10%
    'exposure_frac': 1/3,     # A5 영상 노출 비율 (법 기준)
    'cps': 197,               # A6 초당 글자 수: 실제 광고 상위 10%
    'video_pt': 5.06,         # A7 영상 글자 크기 (가로 화면): 실제 광고 하위 10%
}
VERDICTS = ['위반', '확인필요', '적합', '해당없음']
checks[['check_id', 'item', 'products', 'media', 'mode', 'group']]

## 3. 상품유형 자동 채우기
주연 님 JSON에는 상품유형이 비어 있는 경우가 많아서, **브랜드 · 제목 · 광고 문구의 키워드**로 정합니다.
- 결과가 `전체`면 상품을 특정하지 못한 것(기업 PR 광고 등)이라 공통 항목만 검사합니다.
- `범위밖`은 신용카드처럼 우리 규정 카드로 다루기 어려운 광고입니다.
- 틀린 게 있으면 `PRODUCT_OVERRIDE`에 직접 적으면 그 값이 우선합니다.

In [ ]:
PRODUCT_OVERRIDE = {
    # 'V112': '투자성',
}

RULES = [   # (키워드 정규식, 상품유형) — 위에서부터 먼저 맞는 것
    (r'ETF|펀드|자산운용|ISA|연금|주식|투자|증권|ETN|ELS|채권', '투자성'),
    (r'보험|생명|화재|해상|손해', '보장성'),
    (r'대출|중금리|신용대출|카드론|담보', '대출성'),
    (r'적금|예금|통장|파킹|정기예', '예금성'),
    (r'카드', '범위밖'),
]

def ad_text(ad, limit=None):
    texts = [p['text'] for p in ad.get('paragraphs', [])]
    texts += [c['text'] for c in ad.get('cards', []) if c.get('kind') == 'crawl']
    s = '\n'.join(texts)
    return s[:limit] if limit else s

def infer_product(ad):
    if ad['ad_id'] in PRODUCT_OVERRIDE:
        return PRODUCT_OVERRIDE[ad['ad_id']], 'override'
    p = ad.get('product')
    if p in ('ETF', '증권'):
        return '투자성', 'json'
    head = ' '.join(str(ad.get(k) or '') for k in ('brand', 'title'))
    for src, text in (('brand/title', head), ('text', ad_text(ad, 3000))):
        for pat, ptype in RULES:
            if re.search(pat, text):
                return ptype, src
    return '전체', 'none'

def load_ad(ad_id):
    kind = 'video' if ad_id.startswith('V') else 'image'
    return json.loads((REPO_DIR / f'data/processed/{kind}/json/{ad_id}.json').read_text(encoding='utf-8'))

ALL_IDS = sorted(p.stem for k in ('image', 'video') for p in (REPO_DIR / f'data/processed/{k}/json').glob('*.json'))
prod = pd.DataFrame([{'ad_id': i, **dict(zip(['product', 'by'], infer_product(load_ad(i)))),
                      'brand': load_ad(i).get('brand')} for i in ALL_IDS])
print(prod['product'].value_counts().to_string())
prod[prod['ad_id'].str.startswith('V')].head(30)

## 4. 광고 정리
LLM에 넘길 광고 정보를 한 덩어리로 만듭니다. 위험고지 문단과 그 밖의 문구(헤드라인·수익 표현 등)를 나눠서 넘깁니다.

In [ ]:
def media_of(ad):
    if 'duration' in ad:
        return ad.get('media') or 'video'
    return ad.get('media') or 'image'

def is_video(ad):
    return 'duration' in ad

def summarize_ad(ad, ptype):
    paras = ad.get('paragraphs', [])
    risk = [p['text'] for p in paras if p['type'] == 'risk_notice']
    risk += [c['text'] for c in ad.get('cards', []) if c.get('type') == 'risk_notice' and c.get('kind') == 'crawl']
    other = [f"[{p['type']}] {p['text']}" for p in paras if p['type'] != 'risk_notice']
    return {
        'ad_id': ad['ad_id'], '상품유형': ptype, '매체': media_of(ad), '브랜드': ad.get('brand'),
        '제목': ad.get('title'), '영상 길이(초)': ad.get('duration'),
        '위험고지 문구': list(dict.fromkeys(risk))[:20],
        '그 밖의 문구': list(dict.fromkeys(other))[:60],
        '측정값': ad.get('metrics', {}),
    }

def applicable(row, ptype, media):
    p = row['products']
    ok_p = p == '전체' or (p.startswith('!') and ptype != p[1:]) or p == ptype
    if ptype == '범위밖':
        ok_p = p == '전체'
    m = row['media']
    ok_m = (m == 'all' or (m == 'video' and media in ('video', 'tv_cf', 'online_video'))
            or (m == 'image' and media in ('image', 'poster', 'banner')) or media in m.split(','))
    return ok_p and ok_m

ad = load_ad('AD01'); ptype, _ = infer_product(ad)
json.dumps(summarize_ad(ad, ptype), ensure_ascii=False)[:600]

## 5. 코드 판정
숫자로 판정할 수 있는 항목은 LLM에 맡기지 않고 코드로 정합니다.
필수 문구(B1 설명서 권유, B3 투자 위험, B5 설명 받을 권리, B6 예금자보호)는 키워드가 있으면 코드로 `적합`, 없으면 LLM이 판정합니다. LLM은 이 결과를 **바꿀 수 없고**, 근거 조항만 붙입니다.

In [ ]:
REVIEW_RE = re.compile(r'(심사필|심의필)\s*제?\s*[\d\-]+\s*호')
DATE_RE = re.compile(r'(20\d\d)[.\-](\d{1,2})[.\-](\d{1,2})')

def code_checks(ad, ptype):
    m, media, out = ad.get('metrics', {}), media_of(ad), {}
    video = is_video(ad)
    text = ad_text(ad)
    n_risk = m.get('n_risk_cards') if video else sum(p['type'] == 'risk_notice' for p in ad.get('paragraphs', []))
    out['A1'] = ('확인필요', f'위험고지 문단 미검출 (n={n_risk})') if not n_risk else ('적합', f'위험고지 문단 {n_risk}개 검출')

    c = m.get('risk_contrast_min')
    out['A2'] = ('확인필요', '명암비 측정 불가(null) — 이미지 확인 필요') if c is None else \
                (('확인필요' if c < TH['contrast'] else '적합'), f'최소 명암비 {c:.2f} (기준 {TH["contrast"]})')
    if not video:
        pt = m.get('mobile_feed', {}).get('risk_median_pt')
        if pt is not None:
            out['A3'] = (('확인필요' if pt < TH['image_pt'] else '적합'), f'위험고지 글자 {pt:.2f}pt (기준 {TH["image_pt"]}pt, 폭 390 환산)')
        r, th = m.get('risk_headline_ratio'), TH['ratio_image']
    else:
        r, th = m.get('risk_headline_ratio_min'), TH['ratio_video']
    if r is not None:
        out['A4'] = (('확인필요' if r < th else '적합'), f'위험고지/헤드라인 크기 비율 {r:.3f} (기준 {th})')

    if video:
        d, e = ad.get('duration') or 0, m.get('risk_exposure_s') or 0
        if d:
            frac = e / d
            out['A5'] = (('적합' if frac >= TH['exposure_frac'] else '확인필요'),
                         f'위험고지 노출 {e:.1f}초 / 영상 {d:.1f}초 = {frac:.0%} (기준 1/3, 미만이면 음성 고지 확인 필요)')
        cps = m.get('risk_cps_max')
        if cps is not None:
            out['A6'] = (('확인필요' if cps > TH['cps'] else '적합'), f'최대 초당 글자 수 {cps:.0f}자 (기준 {TH["cps"]}자)')
        vp = (m.get('risk_display_min_pt') or {}).get('mobile_landscape')
        if vp is not None:
            out['A7'] = (('확인필요' if vp < TH['video_pt'] else '적합'), f'가로 화면 기준 최소 글자 {vp:.2f}pt (기준 {TH["video_pt"]}pt)')
    if media == 'banner':
        out['A8'] = ('해당없음', '정지 이미지 배너 — 위험고지가 계속 표시됨')

    review = ad.get('review_no') or [x.group(0) for x in REVIEW_RE.finditer(text)]
    if review:
        out['D1'] = ('적합', f'심사필 표시 검출: {review[0]}')
    elif media == 'tv_cf':
        out['D1'] = ('해당없음', '방송 광고 — 공간 제약 시 심사필 표시 생략 가능')
    else:
        out['D1'] = ('확인필요', '심사필 번호 미검출 (OCR 누락 가능)')
    dates = [datetime.date(int(y), int(mo), int(dd)) for y, mo, dd in DATE_RE.findall(text) if 1 <= int(mo) <= 12 and 1 <= int(dd) <= 31]
    if review and dates:
        end = max(dates)
        out['D2'] = (('적합' if end >= TODAY else '확인필요'), f'표시된 가장 늦은 날짜 {end} (판정일 {TODAY}, 게시일 정보 없음)')
    else:
        out['D2'] = ('확인필요', '유효기간 날짜 미검출')

    # 필수 문구 키워드 선검사: 광고 전체 텍스트(공백 제거)에서 찾고, 있으면 적합. 없으면 LLM이 판정
    flat = re.sub(r'\s+', '', text)
    has = lambda *ks: [k for k in ks if k in flat]
    if has('설명서') and has('필독', '읽어'):
        out['B1'] = ('적합', '키워드 검출: ' + ', '.join(has('설명서', '필독', '읽어')))
    if ptype == '투자성':
        hit = has('원금손실') or (['손실', '투자자'] if has('손실') and has('투자자') else [])   # '원금 손실'은 공백 제거로 포함
        if hit:
            out['B3'] = ('적합', '키워드 검출: ' + ', '.join(hit))
    if has('예금자보호'):
        out['B6'] = ('적합', '키워드 검출: 예금자보호')
    if has('충분한설명', '설명청취'):
        out['B5'] = ('적합', '키워드 검출: ' + ', '.join(has('충분한설명', '설명청취')))
    return out

code_checks(ad, ptype)

## 6. 근거 카드 검색
LLM에 묻는 묶음(AD / B / CE)마다, 그 묶음의 항목별 질문으로 카드를 8장씩 찾고 합칩니다. 검사기준표의 **근거 카드**도 항상 넣어 줍니다(검색이 놓쳐도 빠지지 않게). 묶음당 최대 16장입니다.

In [ ]:
def ref_ids(s):
    ids = []
    for tok in re.split(r'[,;]', s):
        tok = re.sub(r'\(.*?\)|참고:', '', tok).strip().strip("'")
        tok = tok.split(' ')[0]
        if tok in card_by_id.index:
            ids.append(tok)
    return ids

def retrieve(rows, ptype, k=8, cap=16):
    score = {}
    for _, r in rows.iterrows():
        for cid in ref_ids(r['ref_cards']):
            score[cid] = score.get(cid, 0) + 100          # 기준표 근거 카드는 항상 포함
        for cid, s in search_ids(r['query'], ptype, k=k):
            score[cid] = score.get(cid, 0) + 1
    return sorted(score, key=score.get, reverse=True)[:cap]

rows = checks[checks['group'] == 'B']
retrieve(rows[rows.apply(lambda r: applicable(r, ptype, media_of(ad)), axis=1)], ptype)

## 7. 프롬프트
LLM에게 주는 규칙의 핵심:
1. **주어진 카드만** 근거로 쓰고, 인용은 카드 본문에 **그대로 있는 문장**만 씁니다.
2. 코드 판정이 있는 항목은 판정을 바꾸지 않고 근거만 붙입니다.
3. 우리 서비스 기준(숫자가 없는 법 기준을 해석한 것)에 걸린 경우는 `위반`이 아니라 `확인필요`입니다.
4. 위반은 법 기준이 명확하고 광고 문구로 확실할 때만. 애매하면 확인필요입니다.
5. 판정 근거가 된 **광고 문구(ad_quote)**도 그대로 옮기게 해서, 코드로 실제 광고에 있는지 확인합니다.
6. 정해진 JSON 형식으로만 답합니다.

In [ ]:
SYSTEM = '''당신은 한국 금융광고 사전 심의를 돕는 검토자입니다.
광고 정보, 검사 항목, 규정 카드를 받아 항목마다 판정합니다.

규칙
1. 판정은 "위반", "확인필요", "적합", "해당없음" 중 하나입니다.
   - 위반: 법 기준이 명확하고 광고가 그것을 어긴 경우
   - 확인필요: 법에 수치가 없어 우리 서비스 기준으로 걸렸거나, 생략이 허용될 수 있거나, 문구가 OCR로 일부만 읽혀 판단이 불확실한 경우
   - 적합: 기준을 충족한 경우
   - 해당없음: 이 광고에 적용되지 않는 항목 (예: 운용실적 표시가 없는데 운용실적 요건 항목)
2. "코드 판정"이 주어진 항목은 그 판정을 그대로 쓰고, 근거 조항과 이유만 작성합니다.
3. 근거는 반드시 아래 "규정 카드" 중에서만 고르고 card_ids에 카드 ID를 적습니다.
4. quote에는 근거 카드 본문에 실제로 있는 문장을 한 글자도 바꾸지 말고 그대로 옮깁니다(40~150자). 없는 문장을 만들지 마세요.
5. 광고 문구는 OCR 결과라 띄어쓰기·오타가 있을 수 있습니다. 뜻이 같으면 포함된 것으로 봅니다.
6. reason에는 광고의 어떤 문구(또는 측정값) 때문에 그렇게 판정했는지 1~3문장으로 씁니다.
7. 같은 뜻의 표현이면 포함된 것으로 봅니다. 예: "(간이)투자설명서·집합투자규약 필독"은 설명서를 읽어볼 것을 권유한 것입니다.
8. "위반"은 (가) 법 기준이 명확하고 (나) 광고 문구로 어긴 것이 확실할 때만 씁니다.
   표현의 강도가 애매하거나(예: "독보적", "대표") 화면 속 예시 이미지의 숫자처럼 해석이 갈리면 "확인필요"입니다.
9. ad_quote에는 판정의 근거가 된 광고 문구를 광고 정보에서 그대로 옮깁니다. 필요한 문구가 없어서 내린 판정이면 빈 문자열("")로 둡니다.
   코드 판정이 주어진 항목은 ad_quote를 빈 문자열("")로 둡니다.
10. 아래 JSON 형식으로만 답합니다.

{"items": [{"check_id": "...", "verdict": "...", "card_ids": ["..."], "quote": "...", "ad_quote": "...", "reason": "..."}],
 "summary": "이 묶음의 판단 요약 1~2문장"}'''

def card_block(cid, maxlen=1500):
    c = card_by_id.loc[cid]
    body = c['본문'] if len(c['본문']) <= maxlen else c['본문'][:maxlen] + '…(이하 생략)'
    return f"### {cid} | {c['문서명']} {c['조']} {c['조 제목']} | 상품유형: {c['상품유형']}\n{body}"

def build_prompt(ad_info, rows, codes, card_ids):
    items = []
    for _, r in rows.iterrows():
        it = {'check_id': r['check_id'], '검사 항목': r['item'], '법 기준': r['law'], '우리 서비스 기준': r['our_rule'],
              '가능한 판정': r['verdicts']}
        if r['check_id'] in codes:
            it['코드 판정'] = {'판정': codes[r['check_id']][0], '근거 측정값': codes[r['check_id']][1]}
        items.append(it)
    return ('# 광고 정보\n' + json.dumps(ad_info, ensure_ascii=False, indent=1) +
            '\n\n# 검사 항목\n' + json.dumps(items, ensure_ascii=False, indent=1) +
            '\n\n# 규정 카드\n' + '\n\n'.join(card_block(c) for c in card_ids) +
            '\n\n위 검사 항목 전부에 대해 JSON으로 답하세요.')

print(SYSTEM[:300])

## 8. LLM 호출
OpenAI와 Claude를 **같은 함수 모양**으로 부릅니다. 그래야 같은 프롬프트로 두 모델을 비교할 수 있습니다.
키가 없는 쪽은 건너뜁니다.

In [ ]:
def _parse_json(text):
    text = text.strip()
    text = re.sub(r'^```(json)?|```$', '', text, flags=re.M).strip()
    start, end = text.find('{'), text.rfind('}')
    return json.loads(text[start:end + 1])

def call_llm(provider, system, user, model=None):
    model = model or MODELS[provider]
    if provider == 'openai':
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model=model, response_format={'type': 'json_object'},
            messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}])
        text, usage = r.choices[0].message.content, (r.usage.prompt_tokens, r.usage.completion_tokens)
    elif provider == 'claude':
        import anthropic
        r = anthropic.Anthropic().messages.create(
            model=model, max_tokens=8000, system=system, messages=[{'role': 'user', 'content': user}])
        text, usage = r.content[0].text, (r.usage.input_tokens, r.usage.output_tokens)
    else:
        raise ValueError(provider)
    return _parse_json(text), {'model': model, 'input_tokens': usage[0], 'output_tokens': usage[1]}

## 9. 근거 검증
LLM 답을 그대로 믿지 않고 코드로 확인합니다. 문제가 있으면 `issues` 열에 적힙니다.
- 판정 값이 정해진 4가지 중 하나인가
- 코드 판정을 바꾸지 않았는가
- 인용한 카드가 이번에 넘겨준 카드인가
- 인용 문장이 그 카드 본문에 **실제로 있는가** (띄어쓰기 무시하고 비교. `…`·`...`·줄바꿈으로 나눈 5자 이상 조각이 모두 인용 카드 원문에 있어야 통과)
- LLM이 옮긴 **광고 문구(ad_quote)가 실제 광고에 있는가** (` / `로 이어 붙였으면 조각마다 확인)
- 카드 원문과 인용 모두 `<개정 …>`, `<신설 …>` 같은 꺾쇠 주석은 지우고 비교

In [ ]:
def _norm(s):
    return re.sub(r'\s+', '', str(s))

QUOTE_SPLIT_RE = re.compile(r'…|\.\.\.|\n')
NOTE_RE = re.compile(r'<[^<>]*>')          # '<개정 2021.4.15.>', '<신설 ...>' 같은 꺾쇠 주석

def _norm_card(s):
    return _norm(NOTE_RE.sub('', str(s)))

def quote_in_cards(quote, card_ids):
    # 인용을 '…', '...', 줄바꿈으로 조각낸 뒤, 공백 제거 후 5자 이상인 조각이 전부
    # 인용한 카드 중 어느 하나의 원문에 있으면 통과
    pieces = [p for p in (_norm_card(s) for s in QUOTE_SPLIT_RE.split(str(quote))) if len(p) >= 5]
    bodies = [_norm_card(card_by_id.loc[c, '본문']) for c in card_ids if c in card_by_id.index]
    return bool(pieces) and all(any(p in b for b in bodies) for p in pieces)

def validate(item, given_ids, codes):
    issues = []
    if item.get('verdict') not in VERDICTS:
        issues.append(f"판정값 이상({item.get('verdict')})")
    cid = item.get('check_id')
    if cid in codes and item.get('verdict') != codes[cid][0]:
        issues.append(f'코드 판정({codes[cid][0]})을 바꿈')
        item['verdict'] = codes[cid][0]
    bad = [c for c in item.get('card_ids', []) if c not in given_ids]
    if bad:
        issues.append(f'주지 않은 카드 인용({",".join(bad)})')
    q = _norm(item.get('quote', ''))
    if item.get('verdict') != '해당없음':
        if not q:
            issues.append('인용 없음')
        elif not quote_in_cards(item.get('quote', ''), item.get('card_ids', [])):
            issues.append('인용 문장이 카드 원문에 없음')
    aq = [p for p in (_norm(s) for s in str(item.get('ad_quote', '')).split(' / ')) if p]   # ' / '로 이은 여러 문구는 조각마다 확인
    if any(p not in AD_TEXT_NORM for p in aq):
        issues.append('광고 문구 인용이 광고에 없음')
    item['issues'] = '; '.join(issues)
    return item

## 10. 광고 1건 판정
`judge_ad('AD01', 'openai')`처럼 부릅니다. 결과는 표로 보여주고 `outputs/{광고ID}__{모델}.json`과 `.csv`에 저장합니다. CSV는 엑셀이나 Google 시트로 열면 잘리지 않고 다 보입니다.

LLM 호출은 묶음당 1번, 광고 1건에 최대 3번입니다. LLM이 `위반`을 낸 묶음은 같은 프롬프트로 한 번 더 물어(자기 일관성 재확인), 2차에도 `위반`일 때만 유지하고 아니면 `확인필요`로 낮춥니다(issues에 `재확인 불일치` 기록, 토큰·재호출 횟수는 결과 JSON에 합산).
C5(혜택·불이익 균형)는 다른 항목 판정이 끝난 뒤 코드로 정합니다. A2~A7·C3 중 확인필요·위반이 하나라도 있으면 `확인필요`, 없으면 `적합`입니다. LLM의 이유는 종합 의견으로 남깁니다.

In [ ]:
GROUP_NAME = {'AD': '표시 방법·심의 표시', 'B': '필수 내용', 'CE': '금지 표현·추천보증'}
C5_FROM = ['A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'C3']   # 이 중 하나라도 확인필요·위반이면 C5 = 확인필요
CODE_AFTER = {'C5'}   # LLM 답 뒤에 코드가 덮어쓰는 항목 (위반 재확인 대상에서 제외)

def judge_ad(ad_id, provider='openai', model=None, verbose=True):
    ad = load_ad(ad_id)
    ptype, by = infer_product(ad)
    media = media_of(ad)
    info = summarize_ad(ad, ptype)
    global AD_TEXT_NORM
    AD_TEXT_NORM = _norm(json.dumps(info, ensure_ascii=False)) + _norm(ad_text(ad))
    codes = code_checks(ad, ptype)
    rows_all = checks[checks.apply(lambda r: applicable(r, ptype, media), axis=1)]
    results, summaries, usage, rechecks = [], {}, [], []

    for g in ['AD', 'B', 'CE']:
        rows = rows_all[(rows_all['group'] == g) & (rows_all['mode'] != 'skip')]
        rows = rows[(rows['mode'] != 'code') | rows['check_id'].isin(codes)]
        if rows.empty:
            continue
        ids = retrieve(rows, ptype)
        prompt = build_prompt(info, rows, codes, ids)
        out, u = call_llm(provider, SYSTEM, prompt, model)
        usage.append(u); summaries[g] = out.get('summary', '')
        got = {it.get('check_id'): it for it in out.get('items', [])}
        # 위반 재확인(자기 일관성): LLM이 판정한 위반이 있으면 같은 프롬프트로 한 번 더 묻고, 2차도 위반일 때만 유지
        llm_vio = [cid for cid in rows['check_id'] if cid not in codes and cid not in CODE_AFTER
                   and (got.get(cid) or {}).get('verdict') == '위반']
        second = {}
        if llm_vio:
            out2, u2 = call_llm(provider, SYSTEM, prompt, model)
            usage.append(u2)
            got2 = {it.get('check_id'): it for it in out2.get('items', [])}
            second = {cid: (got2.get(cid) or {}).get('verdict', '응답없음') for cid in llm_vio}
        for cid in rows['check_id']:
            it = got.get(cid) or {'check_id': cid, 'verdict': codes.get(cid, ('확인필요',))[0],
                                  'card_ids': [], 'quote': '', 'ad_quote': '', 'reason': 'LLM 답변에 이 항목이 없음'}
            it = validate(it, ids, codes)
            it['source'] = 'code' if cid in codes else 'llm'
            it['measure'] = codes.get(cid, ('', ''))[1]
            it['recheck'] = ''
            if cid in second:
                v2 = second[cid]
                if v2 == '위반':
                    it['recheck'] = '1차 위반→2차 위반(유지)'
                else:
                    it['verdict'] = '확인필요'
                    it['recheck'] = f'1차 위반→2차 {v2}(확인필요로 변경)'
                    it['issues'] = '; '.join(x for x in [it.get('issues', ''), f'재확인 불일치(1차 위반→2차 {v2})'] if x)
                rechecks.append({'group': g, 'check_id': cid, 'first': '위반', 'second': v2, 'final': it['verdict']})
            results.append(it)

    # C5 혜택·불이익 균형: 다른 항목 판정 뒤 코드로 정함. LLM 판정은 덮어쓰고 reason은 종합 의견으로 남김
    final = {it['check_id']: it for it in results}
    if 'C5' in final:
        bad = [f"{c}({final[c]['verdict']})" for c in C5_FROM if c in final and final[c]['verdict'] in ('확인필요', '위반')]
        it = final['C5']
        llm_verdict = it.get('verdict')
        it['verdict'] = '확인필요' if bad else '적합'
        it['source'] = 'code'
        it['measure'] = ('확인필요/위반 항목: ' + ', '.join(bad)) if bad else 'A2~A7·C3에 확인필요/위반 없음'
        it['reason'] = f"[LLM 종합 의견 · LLM 판정 {llm_verdict}] {it.get('reason', '')}"

    res = pd.DataFrame(results).merge(checks[['check_id', 'item']], on='check_id')
    res = res.reindex(columns=['check_id', 'item', 'verdict', 'source', 'measure', 'ad_quote', 'reason', 'card_ids', 'quote', 'issues', 'recheck']).fillna('')
    tok = (sum(u['input_tokens'] for u in usage), sum(u['output_tokens'] for u in usage))
    record = {'ad_id': ad_id, 'product': ptype, 'product_by': by, 'media': media,
              'model': usage[0]['model'] if usage else model, 'tokens': tok, 'llm_calls': len(usage), 'recheck_calls': len({r['group'] for r in rechecks}),
              'rechecks': rechecks, 'summaries': summaries,
              'items': res.to_dict('records'), 'judged_at': str(datetime.datetime.now())}
    name = f"{ad_id}__{record['model']}.json".replace('/', '_')
    (OUT_DIR / name).write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding='utf-8')
    res.to_csv(OUT_DIR / name.replace('.json', '.csv'), index=False, encoding='utf-8-sig')   # 엑셀·시트로 열어 보기용
    if verbose:
        print(f"== {ad_id} | {ad.get('brand')} | {ptype}({by}) | {media} | {record['model']} | 토큰 입력 {tok[0]:,} / 출력 {tok[1]:,} | 위반 재확인 호출 {record['recheck_calls']}회")
        print(res['verdict'].value_counts().to_string())
        for g, s in summaries.items():
            print(f'[{GROUP_NAME[g]}] {s}')
    return res

judge_ad('AD01', 'openai')

위반·확인필요 항목만 보기:

In [ ]:
r = judge_ad('V101', 'openai')
r[r['verdict'].isin(['위반', '확인필요'])]

## 11. 여러 광고 · 여러 모델 비교
최종 비교 때 씁니다. 같은 광고 목록을 모델마다 돌리고, **항목별 판정이 얼마나 일치하는지**와 **근거 검증 통과율**을 봅니다.
정답표(위반 예시)가 생기면 여기에 정확도 계산을 붙입니다.

In [ ]:
def run_many(ad_ids, providers=('openai', 'claude')):
    rows = []
    for p in providers:
        key = 'OPENAI_API_KEY' if p == 'openai' else 'ANTHROPIC_API_KEY'
        if not os.getenv(key):
            print(f'{p}: 키 없음 — 건너뜀'); continue
        for a in ad_ids:
            r = judge_ad(a, p, verbose=False)
            r['ad_id'], r['provider'] = a, p
            rows.append(r)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

# 예시: 실제 ETF 광고 5건
# many = run_many(['AD01', 'AD02', 'V101', 'V102', 'V113'])
# many.pivot_table(index=['ad_id', 'check_id'], columns='provider', values='verdict', aggfunc='first')
# many.groupby('provider')['issues'].apply(lambda s: (s == '').mean())   # 근거 검증 통과율